In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd drive/MyDrive/multimodal-rag/

/content/drive/MyDrive/multimodal-rag


In [4]:
!pip install -q -U transformers accelerate qwen-vl-utils peft bert_score rouge_score nltk

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 68.1 MB/s eta 0:00:00


True

In [5]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of sglang to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of sglang to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 62.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metada

In [9]:
!pip install --upgrade torchao


  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import os
import sys
import json
import gc
import torch
import pandas as pd
from tqdm.notebook import tqdm

REPO_ROOT = '/content/drive/MyDrive/multimodal-rag'
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from scripts.vlm import VLM
from scripts.evaluation import evaluate_dataframe

## Config

In [3]:
# Paths (relative to REPO_ROOT)
QAS_PATH = 'accepted/qas.jsonl'
RESULTS_DIR = 'results/lecture_slides_finetune_compare'

# Base model — downloaded to /content/ to keep Drive clean
BASE_MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
BASE_LOCAL_DIR = '/content/Qwen3-VL-8B-Instruct'

# LoRA adapter for the finetuned variant
ADAPTER_PATH = os.path.join(REPO_ROOT, 'finetuned_qwen_adapter/checkpoint-350')

# Generation cap — slides eval expects short answers, no need for 1024
MAX_NEW_TOKENS = 256

os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
adapter_files = os.listdir(ADAPTER_PATH)
print(f"Files in {ADAPTER_PATH}:")
for fn in sorted(adapter_files):
    print(f"  {fn}")

is_lora = 'adapter_config.json' in adapter_files
is_merged = any(fn.startswith('model-') and fn.endswith('.safetensors') for fn in adapter_files)
print(f"\nDetected: LoRA adapter = {is_lora}, merged model = {is_merged}")


Files in /content/drive/MyDrive/multimodal-rag/finetuned_qwen_adapter/checkpoint-350:
  README.md
  adapter_config.json
  adapter_model.safetensors
  optimizer.pt
  rng_state.pth
  scheduler.pt
  trainer_state.json
  training_args.bin

Detected: LoRA adapter = True, merged model = False


## Load QAs

In [5]:
from json import JSONDecoder

decoder = JSONDecoder()
with open(QAS_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

records = []
i = 0
n = len(text)
while i < n:
    # Skip whitespace (including newlines) between objects
    while i < n and text[i].isspace():
        i += 1
    if i >= n:
        break
    obj, end = decoder.raw_decode(text, i)
    records.append({
        'example_index': obj.get('example_index'),
        'question': obj.get('question'),
        'answer': obj.get('answer'),
        'intended_img': obj.get('image_path'),
    })
    i = end

df = pd.DataFrame(records)
print(f"Loaded {len(df)} QAs across {df['intended_img'].nunique()} images")
df.head(3)

Loaded 671 QAs across 650 images


,example_index,question,answer,intended_img
0,0,How does the dialogue construct a number Q to ...,The dialogue constructs Q by multiplying all p...,accepted/cd2ee089_p0020.png
1,1,What is the primary focus of the content prese...,The primary focus is to define and explain wha...,accepted/cd2ee089_p0001.png
2,2,What does the probability expression p(x_t+1 |...,It represents the probability of generating th...,accepted/cd2ee089_p0035.png


## Inference

In [6]:
SYSTEM_PROMPT = (
    "You are a question answering assistant for lecture content. "
    "Respond in one sentence using all available information."
)

def run_inference(vlm, df, model_label, output_path):
    """Run inference on every row, write JSONL incrementally, return the path."""
    with open(output_path, 'w', encoding='utf-8') as f:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Inference [{model_label}]"):
            messages = [
                {
                    'role': 'system',
                    'content': [{'type': 'text', 'text': SYSTEM_PROMPT}],
                },
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': row['intended_img'], 'max_pixels': 1003520},
                        {'type': 'text', 'text': row['question']},
                    ],
                },
            ]

            try:
                response = vlm.generate(messages, sampling_params={'max_new_tokens': MAX_NEW_TOKENS})
                generated = response['text']
            except Exception as e:
                print(f"  [skip] example_index={row['example_index']}: {e}")
                continue

            f.write(json.dumps({
                'dataset': 'LectureSlides',
                'model': model_label,
                'example_index': row['example_index'],
                'question': row['question'],
                'generated_answer': generated,
                'ground_truth': row['answer'],
                'intended_img': row['intended_img'],
            }) + '\n')
            f.flush()

    return output_path

## Base VLM

In [11]:
vlm_base = VLM(model_id=BASE_MODEL_ID, checkpoint_path=BASE_LOCAL_DIR)

base_results_path = os.path.join(RESULTS_DIR, 'base_results.jsonl')
run_inference(vlm_base, df, model_label='base', output_path=base_results_path)
print(f"Base results -> {base_results_path}")

--- Downloading Qwen/Qwen3-VL-8B-Instruct to: /content/Qwen3-VL-8B-Instruct ---


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

--- flash_attention_2 unavailable (ImportError); using sdpa ---


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Inference [base]:   0%|          | 0/671 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Base results -> results/lecture_slides_finetune_compare/base_results.jsonl


In [10]:
del vlm_base
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.1f} / {total/1e9:.1f} GB")

NameError: name 'vlm_base' is not defined

**Restart the Notebook**

## Finetuned VLM

In [11]:
vlm_ft = VLM(
    model_id=BASE_MODEL_ID,
    checkpoint_path=BASE_LOCAL_DIR,
    adapter_path=ADAPTER_PATH if is_lora else None,
)

ft_results_path = os.path.join(RESULTS_DIR, 'finetuned_results.jsonl')
run_inference(vlm_ft, df, model_label='finetuned', output_path=ft_results_path)
print(f"Finetuned results -> {ft_results_path}")

--- Local model found at: /content/Qwen3-VL-8B-Instruct ---
--- Using attn_implementation=sdpa ---


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

--- Loading adapter from: /content/drive/MyDrive/multimodal-rag/finetuned_qwen_adapter/checkpoint-350 ---


Inference [finetuned]:   0%|          | 0/671 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Finetuned results -> results/lecture_slides_finetune_compare/finetuned_results.jsonl


In [ ]:
del vlm_ft
gc.collect()
torch.cuda.empty_cache()

## Evaluation

In [ ]:
def load_results(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

base_df = load_results(base_results_path)
ft_df = load_results(ft_results_path)

print(f"Base:      {len(base_df)} rows")
print(f"Finetuned: {len(ft_df)} rows")

In [ ]:
# Score base
base_report = os.path.join(RESULTS_DIR, 'base_eval_report.txt')
base_eval_df = evaluate_dataframe(base_df.copy(), base_report)
base_eval_df.to_json(os.path.join(RESULTS_DIR, 'base_eval_details.jsonl'),
                    orient='records', lines=True)

In [ ]:
# Score finetuned
ft_report = os.path.join(RESULTS_DIR, 'finetuned_eval_report.txt')
ft_eval_df = evaluate_dataframe(ft_df.copy(), ft_report)
ft_eval_df.to_json(os.path.join(RESULTS_DIR, 'finetuned_eval_details.jsonl'),
                   orient='records', lines=True)